In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
class CustomMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim//num_heads

        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)

        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.size()
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        Q_h = Q.reshape(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)   #B, num_heads, T, d
        K_h = K.reshape(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)   #B, num_heads, T, d
        V_h = V.reshape(B, T, self.num_heads, self.head_dim).permute(0,2,1,3)   #B, num_heads, T, d
        scores = (Q_h@K_h.permute(0,1,3,2))/math.sqrt(self.head_dim)    #B, num_heads, T, T
        attention_weights = F.softmax(scores, dim=-1)   #B, H, T, T
        A = attention_weights@V_h   #B, H, T, D
        A = A.permute(0,2,1,3).contiguous().view(B, T, self.num_heads*self.head_dim)
        self.out_proj(A)
        return A

# --- ТЕСТИРОВАНИЕ ---
# Если код написан верно, размерность на выходе совпадет с размерностью на входе
torch.manual_seed(42)
B, T, C = 2, 8, 512 # 2 текста, 8 токенов в каждом, 512 размерность эмбеддинга
x = torch.randn(B, T, C)

mha = CustomMultiHeadAttention(embed_dim=C, num_heads=8)
out = mha(x)

print(f"Входной shape: {x.shape}")
print(f"Выходной shape: {out.shape}")
assert x.shape == out.shape, "Ошибка: размерности не совпадают!"
print("Успех! Attention вычислен верно.")

Входной shape: torch.Size([2, 8, 512])
Выходной shape: torch.Size([2, 8, 512])
Успех! Attention вычислен верно.
